In [ ]:
# default_exp paper.plsr.validation_curve

# 3.2. Validation Curve (PLSR)

> Computing the validation curve of PLSR for the prediction of exchangeable potassium (K ex.)

In our experiment, the Mean Square Error (MSE) of model prediction on both the training and validation test are computed as model capacity increases (here the number of PLSR components);

In [ ]:
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive',  force_remount=False)
    !pip install mirzai
else:
    %load_ext autoreload
    %autoreload 2

In [ ]:
# mirzai utilities
from mirzai.data.loading import load_kssl
from mirzai.data.selection import (select_y, select_tax_order, select_X)
from mirzai.data.transform import (log_transform_y, SNV, TakeDerivative,
                                   DropSpectralRegions, CO2_REGION)

from mirzai.training.plsr import compute_valid_curve

from fastcore.transform import compose

# Python utilities
from collections import OrderedDict
from tqdm.auto import tqdm
from pathlib import Path

# Data science stack
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

## Load and transform

In [ ]:
src_dir = '../_data'
fnames = ['spectra-features.npy', 'spectra-wavenumbers.npy', 
          'depth-order.npy', 'target.npy', 
          'tax-order-lu.pkl', 'spectra-id.npy']

X, X_names, depth_order, y, tax_lookup, X_id = load_kssl(src_dir, fnames=fnames)

data = X, y, X_id, depth_order

transforms = [select_y, select_tax_order, select_X, log_transform_y]
X, y, X_id, depth_order = compose(*transforms)(data)

In [ ]:
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Wavenumbers:\n {X_names}')
print(f'depth_order (first 3 rows):\n {depth_order[:3, :]}')
print(f'Taxonomic order lookup:\n {tax_lookup}')

X shape: (40132, 1764)
y shape: (40132,)
Wavenumbers:
 [3999 3997 3995 ...  603  601  599]
depth_order (first 3 rows):
 [[43.  2.]
 [ 0.  0.]
 [ 0.  1.]]
Taxonomic order lookup:
 {'alfisols': 0, 'mollisols': 1, 'inceptisols': 2, 'entisols': 3, 'spodosols': 4, 'undefined': 5, 'ultisols': 6, 'andisols': 7, 'histosols': 8, 'oxisols': 9, 'vertisols': 10, 'aridisols': 11, 'gelisols': 12}


## Experiment

### Setup

In [ ]:
#pls_components = range(5, 120, 5) # List of PLSR components to try
pls_components = range(5, 20, 5) # List of PLSR components to try
#seeds_range = range(20) # List of random seeds to use for multiple train/test splits 
seeds_range = range(2) # List of random seeds to use for multiple train/test splits 
test_size = 0.1 # Train/test split ratio

### Run

#### On all Soil Taxonomy Orders

In [ ]:
# On all data over different random seeds
history = compute_valid_curve(X, 
                              y,
                              X_names,
                              mask=None,
                              pls_components=pls_components,
                              seeds=seeds_range,
                              test_size=test_size)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
history

OrderedDict([('pls_components', range(5, 20, 5)),
             ('train_score',
              [[0.08449216372759856, 0.0704171216398762, 0.06413505876858068],
               [0.0848021976407181,
                0.07088075113138467,
                0.06456415300558434]]),
             ('valid_score',
              [[0.08834169371455501, 0.07441504115862985, 0.0685760184050537],
               [0.08433350155495287,
                0.0706297984855506,
                0.06552652361383798]])])

#### On a specific Soil Taxonomy Orders

In [ ]:
# A quick look at the Soil Taxonomy Orders lookup table
tax_lookup

{'alfisols': 0,
 'mollisols': 1,
 'inceptisols': 2,
 'entisols': 3,
 'spodosols': 4,
 'undefined': 5,
 'ultisols': 6,
 'andisols': 7,
 'histosols': 8,
 'oxisols': 9,
 'vertisols': 10,
 'aridisols': 11,
 'gelisols': 12}

In [ ]:
mask = (depth_order[:, 1] == 1) # Select only mollisols for instance 

In [ ]:
# On all data over different random seeds
compute_valid_curve(X, 
                    y,
                    X_names,
                    mask=mask,
                    pls_components=pls_components,
                    seeds=seeds_range,
                    test_size=test_size)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

OrderedDict([('pls_components', range(5, 20, 5)),
             ('train_score',
              [[0.05528403409560565,
                0.04461938665960243,
                0.038708822647032244],
               [0.05548779501590159,
                0.04490850071906099,
                0.039041544985461245]]),
             ('valid_score',
              [[0.054688076614910645,
                0.04439372865452716,
                0.039963833531820946],
               [0.05451586368447427,
                0.04126731203673031,
                0.0369554952225429]])])

### Save results

In [ ]:
dest_dir = 'name_of_your_destination_directory'

with open(Path(dest_dir)/'history_pls_validation_curve.pickle', 'wb') as f: 
    pickle.dump(history, f)